In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar el archivo CSV con las coordenadas y etiquetas
csv_file = 'detecciones_con_posicion.csv'
df = pd.read_csv(csv_file)

# Filtrar los datos de las posiciones (Izquierda o Derecha)
# Convertir las etiquetas a valores numéricos (0: Izquierda, 1: Derecha)
df['posicion'] = df['posicion'].apply(lambda x: 0 if x == 'Izquierda del rayo' else 1)

# Extraer las características (coordenadas de base y rayo)
X = df[['x1_base', 'y1_base', 'x2_base', 'y2_base', 'x1_rayo', 'y1_rayo', 'x2_rayo', 'y2_rayo']].values

# Etiquetas (Izquierda: 0, Derecha: 1)
y = df['posicion'].values

# Normalizar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convertir las etiquetas a formato categórico (si es necesario)
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# Paso 2: Construcción del modelo de red neuronal

model = Sequential()

# Capa de entrada (con 8 características de entrada)
model.add(Dense(64, input_dim=8, activation='relu'))

# Capa oculta
model.add(Dense(32, activation='relu'))

# Capa de salida (2 clases: Izquierda, Derecha)
model.add(Dense(2, activation='softmax'))

# Compilación del modelo
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Paso 3: Entrenamiento del modelo
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Accuracy en el conjunto de prueba: {accuracy*100:.2f}%')

# Guardar el modelo entrenado
model.save('modelo_rayo_derecha_izquierda.h5')

joblib.dump(scaler, "scaler2.pkl")


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6415 - loss: 0.6359 - val_accuracy: 0.7128 - val_loss: 0.5704
Epoch 2/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7376 - loss: 0.5455 - val_accuracy: 0.7340 - val_loss: 0.5505
Epoch 3/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6651 - loss: 0.5710 - val_accuracy: 0.7553 - val_loss: 0.5229
Epoch 4/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7225 - loss: 0.5247 - val_accuracy: 0.7553 - val_loss: 0.5080
Epoch 5/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7225 - loss: 0.4945 - val_accuracy: 0.7553 - val_loss: 0.4936
Epoch 6/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7497 - loss: 0.4847 - val_accuracy: 0.7979 - val_loss: 0.4695
Epoch 7/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7674 - loss: 0.4749 - val_accuracy: 0.8085 - val_loss: 0.4519
Epoch 8/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7687 - loss: 0.4447 - val_accuracy: 0.8404 - val_loss: 0.4338
Ep

Accuracy en el conjunto de prueba: 97.87%


['scaler2.pkl']

In [ ]:
import csv

# Función para determinar la posición de la base con respecto al rayo
def determinar_posicion(base, rayo):
    """
    Determina si la base está más cerca de la izquierda o derecha del bounding box del rayo.
    """
    # Calcular el centro de la base y del rayo
    centro_base = (base[0] + base[2]) / 2  # Centro de la base (x1_base, x2_base)
    centro_rayo = (rayo[0] + rayo[2]) / 2   # Centro del rayo (x1_rayo, x2_rayo)

    # Comparar las posiciones
    if centro_base < centro_rayo:
        return "Izquierda del rayo"
    elif centro_base > centro_rayo:
        return "Derecha del rayo"
    else:
        return "Sobre el centro del rayo"

# Función para cargar el CSV de entrada, procesar los datos y guardarlos en un nuevo archivo CSV
def procesar_csv(input_csv, output_csv):
    with open(input_csv, mode="r") as infile, open(output_csv, mode="w", newline="") as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        # Escribir los encabezados en el archivo de salida
        writer.writerow(["timestamp", "class_base", "x1_base", "y1_base", "x2_base", "y2_base",
                         "class_rayo", "x1_rayo", "y1_rayo", "x2_rayo", "y2_rayo", "posicion"])

        # Saltar los encabezados del archivo CSV de entrada
        next(reader)

        # Leer y procesar cada fila del archivo CSV
        for row in reader:
            timestamp = row[0]
            class_base = row[1]
            x1_base, y1_base, x2_base, y2_base = map(int, row[2:6])  # Coordenadas de la base
            class_rayo = row[6]
            x1_rayo, y1_rayo, x2_rayo, y2_rayo = map(int, row[7:11])  # Coordenadas del rayo

            # Determinar la posición (izquierda o derecha) de la base con respecto al rayo
            posicion = determinar_posicion((x1_base, y1_base, x2_base, y2_base),
                                           (x1_rayo, y1_rayo, x2_rayo, y2_rayo))

            # Escribir la fila con la nueva información (posicion)
            writer.writerow([timestamp, class_base, x1_base, y1_base, x2_base, y2_base,
                             class_rayo, x1_rayo, y1_rayo, x2_rayo, y2_rayo, posicion])

# Llamar a la función para procesar el CSV
input_csv = 'deteccionesrayo1.0.csv'  # Nombre del archivo de entrada
output_csv = 'detecciones_con_posicion.csv'  # Nombre del archivo de salida
procesar_csv(input_csv, output_csv)

print(f"Los datos procesados se han guardado en {output_csv}")


Los datos procesados se han guardado en detecciones_con_posicion.csv
